In [18]:
#Cell 1 - Download requirements
!pip install datasets anthropic requests python-dotenv matplotlib -q



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# ── Cell 2 — Configuration ────────────────────────────────────────────────────
from dotenv import load_dotenv
import os, anthropic

MODEL      = "claude-sonnet-4-6"
MAX_TOKENS = 20000
DATASET    = "princeton-nlp/SWE-bench_Verified"

BATCH = 0   # 0 = first batch, 1 = second batch, etc.

# Sonnet 4.6 pricing (per million tokens)
INPUT_COST_PER_M  = 3.00
OUTPUT_COST_PER_M = 15.00

SHARED_INSTANCES_FILE = "pilot_instances.json"
PREDICTIONS_FILE      = f"predictions_hctg_batch{BATCH}.jsonl"
EXTRACTED_FILE        = f"predictions_hctg_batch{BATCH}_extracted.jsonl"
FIXED_FILE            = f"predictions_hctg_batch{BATCH}_fixed.jsonl"
RESULTS_FILE          = f"results_hctg_batch{BATCH}.json"

RUN_ID_BASE           = f"hctg_batch{BATCH}"

load_dotenv(override=True)
client = anthropic.Anthropic()


In [20]:
# ── Cell 3 — Load dataset + pilot instances ───────────────────────────────────
import json
from datasets import load_dataset

ds        = load_dataset(DATASET, split="test")
ds_lookup = {inst["instance_id"]: inst for inst in ds}

with open(SHARED_INSTANCES_FILE) as f:
    all_batches = json.load(f)  # list of batches, each a list of instance_ids

pilot_instances = [ds_lookup[iid] for iid in all_batches[BATCH] if iid in ds_lookup]
inst_lookup     = {i["instance_id"]: i for i in pilot_instances}

print(f"Batch {BATCH}: {len(pilot_instances)} instances")


Batch 0: 50 instances


In [21]:
# ── Cell 4 — Repo utilities ────────────────────────────────────────────────────
import subprocess, os, ast

# Root folder where all repos are cloned, e.g. repos/django, repos/astropy
REPOS_BASE = "repos"

# Maps the org prefix of an instance_id to its local clone folder name
ORG_TO_DIR = {
    "astropy"     : "astropy",
    "django"      : "django",
    "matplotlib"  : "matplotlib",
    "pylint-dev"  : "pylint",
    "pytest-dev"  : "pytest",
    "scikit-learn": "scikit-learn",
    "sphinx-doc"  : "sphinx",
    "sympy"       : "sympy",
    "pydata"      : "xarray",
}

def get_repo_dir(iid):
    """Return the local clone path for an instance, e.g. 'repos/django'."""
    org = iid.split("__")[0]
    return os.path.join(REPOS_BASE, ORG_TO_DIR.get(org, org))

def get_file_tree(repo_dir, commit):
    """Return all file paths in the repo at a given commit as a newline-separated string."""
    r = subprocess.run(
        ["git", "ls-tree", "-r", "--name-only", commit],
        capture_output=True, text=True, cwd=repo_dir, encoding="utf-8",
    )
    return r.stdout.strip() if r.returncode == 0 else ""

def get_file_content(repo_dir, commit, filepath):
    """Return the content of a single file at a given commit, or None if it doesn't exist."""
    r = subprocess.run(
        ["git", "show", f"{commit}:{filepath}"],
        capture_output=True, text=True, cwd=repo_dir, encoding="utf-8", errors="replace",
    )
    return r.stdout.replace("\r\n", "\n").replace("\r", "\n") if r.returncode == 0 else None

def generate_ast_skeleton(source_code, file_path=""):
    """
    Parse Python source and return a compact structural skeleton:
    imports, module-level constants, class definitions with method
    signatures and docstrings, and top-level functions.
    """
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return f"# FILE: {file_path}\n# [SYNTAX ERROR — could not parse]\n"

    lines = []
    if file_path:
        lines.append(f"# FILE: {file_path}")
        lines.append("")

    def get_docstring(node):
        ds = ast.get_docstring(node)
        return ds.strip().split("\n")[0][:120] if ds else None

    def format_args(args_node):
        parts = []
        defaults_offset = len(args_node.args) - len(args_node.defaults)
        for i, arg in enumerate(args_node.args):
            annotation = ""
            if arg.annotation:
                try:
                    annotation = ": " + ast.unparse(arg.annotation)
                except Exception:
                    pass
            default = ""
            default_idx = i - defaults_offset
            if 0 <= default_idx < len(args_node.defaults):
                try:
                    default = "=" + ast.unparse(args_node.defaults[default_idx])
                    if len(default) > 20:
                        default = "=..."
                except Exception:
                    default = "=..."
            parts.append(f"{arg.arg}{annotation}{default}")
        if args_node.vararg:
            parts.append(f"*{args_node.vararg.arg}")
        elif args_node.kwonlyargs:
            parts.append("*")
        for i, arg in enumerate(args_node.kwonlyargs):
            default = ""
            if i < len(args_node.kw_defaults) and args_node.kw_defaults[i] is not None:
                try:
                    default = "=" + ast.unparse(args_node.kw_defaults[i])
                    if len(default) > 20:
                        default = "=..."
                except Exception:
                    default = "=..."
            parts.append(f"{arg.arg}{default}")
        if args_node.kwarg:
            parts.append(f"**{args_node.kwarg.arg}")
        return ", ".join(parts)

    def format_function(node, indent=0):
        prefix = "    " * indent
        decorators = ""
        for dec in node.decorator_list:
            try:
                dec_name = ast.unparse(dec)
                decorators += f"{prefix}@{dec_name[:40]}\n"
            except Exception:
                pass
        ret = ""
        if node.returns:
            try:
                ret = " -> " + ast.unparse(node.returns)
            except Exception:
                pass
        result = f"{decorators}{prefix}def {node.name}({format_args(node.args)}){ret}:"
        ds = get_docstring(node)
        if ds:
            result += f'\n{prefix}    "{ds}"'
        return result

    # Imports (deduplicated, max 15)
    imports = []
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.append(f"import {alias.name}")
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            names = ", ".join(a.name for a in node.names[:5])
            if len(node.names) > 5:
                names += ", ..."
            imports.append(f"from {module} import {names}")
    if imports:
        seen, unique = set(), []
        for imp in imports:
            if imp not in seen:
                seen.add(imp)
                unique.append(imp)
        lines.extend(unique[:15])
        if len(unique) > 15:
            lines.append(f"# ... and {len(unique) - 15} more imports")
        lines.append("")

    # Module-level constants
    module_vars = []
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                try:
                    name = ast.unparse(target)
                    if name.isupper() or name.startswith("__"):
                        module_vars.append(name)
                except Exception:
                    pass
    if module_vars:
        lines.append(f"# Module-level constants: {', '.join(module_vars[:10])}")
        lines.append("")

    # Classes and top-level functions
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.ClassDef):
            bases = ""
            if node.bases:
                try:
                    bases = "(" + ", ".join(ast.unparse(b) for b in node.bases) + ")"
                except Exception:
                    bases = "(...)"
            lines.append(f"class {node.name}{bases}:")
            ds = get_docstring(node)
            if ds:
                lines.append(f'    "{ds}"')
            class_attrs = []
            for child in ast.iter_child_nodes(node):
                if isinstance(child, ast.Assign):
                    for target in child.targets:
                        try:
                            class_attrs.append(ast.unparse(target))
                        except Exception:
                            pass
            if class_attrs:
                lines.append(f"    # Attributes: {', '.join(class_attrs[:8])}")
            for method in [c for c in ast.iter_child_nodes(node) if isinstance(c, (ast.FunctionDef, ast.AsyncFunctionDef))]:
                lines.append(format_function(method, indent=1))
            lines.append("")
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            lines.append(format_function(node, indent=0))
            lines.append("")

    return "\n".join(lines)

print("Repo utilities ready")


Repo utilities ready


In [22]:
# ── Cell 5 — Cost calculator ──────────────────────────────────────────────────

def compute_cost(usage):
    """Convert a response's usage object into token counts and USD cost."""
    input_cost  = usage.input_tokens  / 1_000_000 * INPUT_COST_PER_M
    output_cost = usage.output_tokens / 1_000_000 * OUTPUT_COST_PER_M
    return {
        "input_tokens"   : usage.input_tokens,
        "output_tokens"  : usage.output_tokens,
        "input_cost_usd" : round(input_cost,  6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd" : round(input_cost + output_cost, 6),
    }


In [23]:
# ── Cell 6 — File localization (Hypothesis → AST skeleton) ────────────────────
# HCTG Stages 1-2: Form expected-flow hypothesis from the issue alone,
# then use it to guide file selection from the AST skeleton index.
import json as _json, re as _re

_SKIP_PATTERNS = (
    "tests/", "test_", "/migrations/", "docs/", "examples/",
    "setup.py", "conftest.py", "manage.py",
)
_MAX_INDEX_FILE_CHARS = 100_000


HYPOTHESIS_SYSTEM = (
    "You are a senior engineer debugging a bug BEFORE reading any source code. "
    "From ONLY the issue description, sketch what the code flow *should* look like "
    "to resolve the reported problem.\n\n"
    "Output format: a markdown adjacency list. One node per line. Nodes are ROLES "
    "(e.g. `parse_request`, `validate_schema`, `apply_default`), not real function "
    "names yet — you have not seen the code. Use `->` for edges. Annotate with "
    "`[passes: X]` for data flow, `[hypothesis]` when uncertain, `??` for suspected "
    "missing behavior.\n\n"
    "Example:\n"
    "  parse_request [hypothesis]\n"
    "    -> validate_schema [passes: raw_dict]\n"
    "    -> ?? missing: default value injection\n\n"
    "Output the graph only. No prose before or after."
)

LOCALIZE_SYSTEM = (
    "You are a software engineer. You have a hypothesis graph describing the "
    "expected flow for a bug, and a structural index of the repository. "
    "Identify which files likely contain the symbols that map to the hypothesis "
    "nodes — these are the files that need to be modified.\n"
    "Return ONLY a JSON object: {\"files\": [\"path/to/file.py\", ...]}, max 4 files, most relevant first.\n"
    "Do not explain. Do not include any text before or after the JSON."
)

LOCALIZE_RETRY_SYSTEM = (
    "You are a software engineer. Output ONLY raw JSON — no explanation, no markdown, no prose.\n"
    "Example of the ONLY acceptable response format:\n"
    "{\"files\": [\"django/db/models/query.py\", \"django/db/models/sql/compiler.py\"]}\n"
    "Now identify which files need to be modified for the following hypothesis and issue."
)


def build_hypothesis(inst):
    """HCTG Stage 1: generate expected-flow graph from issue alone."""
    user_msg = f"<issue>\n{inst['problem_statement']}\n</issue>"
    resp = client.messages.create(
        model=MODEL, max_tokens=1500, temperature=0.0,
        system=HYPOTHESIS_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    return resp.content[0].text.strip(), compute_cost(resp.usage)["total_cost_usd"]


def build_skeleton_index(repo_dir, commit):
    """Build AST skeleton of every relevant .py file at base_commit."""
    tree = get_file_tree(repo_dir, commit)
    if not tree:
        return ""
    py_files = [
        p for p in tree.splitlines()
        if p.endswith(".py") and not any(pat in p for pat in _SKIP_PATTERNS)
    ]
    skeletons = []
    for path in py_files:
        content = get_file_content(repo_dir, commit, path)
        if not content or len(content) > _MAX_INDEX_FILE_CHARS:
            continue
        skeletons.append(generate_ast_skeleton(content, path))
    return "\n".join(skeletons)


def parse_files_from_response(text):
    """Extract file list from model response — handles JSON, fenced JSON, prose fallback."""
    try:
        return _json.loads(text).get("files", [])
    except Exception:
        pass
    cleaned = _re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    try:
        return _json.loads(cleaned).get("files", [])
    except Exception:
        pass
    return _re.findall(r'"([^"\s]+\.[a-z]+)"', text)


def localize(inst):
    """
    Hypothesis-driven localization:
      1. Form expected-flow hypothesis from issue + failing tests.
      2. Build AST skeleton index of the repo.
      3. Ask the model to ground hypothesis nodes to real files in the index.
      4. Validate each path exists at base_commit; retry once on failure.
    Returns (valid_files, hypothesis, cost, tag).
    """
    bc       = inst["base_commit"]
    repo_dir = get_repo_dir(inst["instance_id"])

    hypothesis, cost = build_hypothesis(inst)

    skeleton_index = build_skeleton_index(repo_dir, bc)
    if not skeleton_index:
        return [], hypothesis, cost, " (no index)"

    user_msg = (
        f"<issue>\n{inst['problem_statement']}\n</issue>\n\n"
        f"<hypothesis_graph>\n{hypothesis}\n</hypothesis_graph>\n\n"
        f"<skeleton_index>\n{skeleton_index}\n</skeleton_index>"
    )

    resp  = client.messages.create(
        model=MODEL, max_tokens=256, temperature=0.0,
        system=LOCALIZE_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    files = parse_files_from_response(resp.content[0].text.strip())
    valid = [fp for fp in files[:4] if get_file_content(repo_dir, bc, fp) is not None]
    cost += compute_cost(resp.usage)["total_cost_usd"]

    if valid:
        return valid, hypothesis, cost, ""

    resp2  = client.messages.create(
        model=MODEL, max_tokens=256, temperature=0.0,
        system=LOCALIZE_RETRY_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    files2 = parse_files_from_response(resp2.content[0].text.strip())
    valid  = [fp for fp in files2[:4] if get_file_content(repo_dir, bc, fp) is not None]
    cost  += compute_cost(resp2.usage)["total_cost_usd"]

    return valid, hypothesis, cost, " (retry)"


# ── Run localization over all pilot instances ──────────────────────────────────
# Cache-aware: if localized + hypotheses files from a previous run exist and
# cover all pilot instances, load them instead of re-running. Delete
# `localized_batch{BATCH}.json` and `hypotheses_hctg_batch{BATCH}.json` to force
# a fresh run (e.g., after changing HYPOTHESIS_SYSTEM or the skeleton index logic).
from pathlib import Path

LOCALIZED_CACHE  = Path(f"localized_batch{BATCH}.json")
HYPOTHESES_CACHE = Path(f"hypotheses_hctg_batch{BATCH}.json")

def _cache_complete():
    if not (LOCALIZED_CACHE.exists() and HYPOTHESES_CACHE.exists()):
        return False
    try:
        loc = _json.load(open(LOCALIZED_CACHE))
        hyp = _json.load(open(HYPOTHESES_CACHE))
    except Exception:
        return False
    return all(i["instance_id"] in loc and i["instance_id"] in hyp for i in pilot_instances)

if _cache_complete():
    localized  = _json.load(open(LOCALIZED_CACHE))
    hypotheses = _json.load(open(HYPOTHESES_CACHE))
    total_loc_cost = 0.0
    print(f"Loaded cached localization + hypotheses for {len(pilot_instances)} instances")
    print(f"  (delete {LOCALIZED_CACHE.name} and {HYPOTHESES_CACHE.name} to force re-run)")
else:
    localized      = {}
    hypotheses     = {}
    total_loc_cost = 0.0

    for i, inst in enumerate(pilot_instances):
        iid      = inst["instance_id"]
        repo_dir = get_repo_dir(iid)

        if not os.path.isdir(repo_dir):
            localized[iid]  = []
            hypotheses[iid] = None
            print(f"[{i+1}/{len(pilot_instances)}] {iid}: repo not found at {repo_dir}")
            continue

        valid, hyp, cost, tag = localize(inst)
        localized[iid]  = valid
        hypotheses[iid] = hyp
        total_loc_cost += cost

        if valid:
            print(f"[{i+1}/{len(pilot_instances)}] {iid}: {valid}{tag}  ${cost:.4f}")
        else:
            print(f"[{i+1}/{len(pilot_instances)}] {iid}: no files found{tag}")

    print(f"\nLocalization done. Total cost: ${total_loc_cost:.4f}")


Loaded cached localization + hypotheses for 50 instances
  (delete localized_batch0.json and hypotheses_hctg_batch0.json to force re-run)


In [24]:
# ── Cell 7 — Localization accuracy ────────────────────────────────────────────
import re as _re

def gold_files(inst):
    """Files touched by the gold patch — ground truth for localization eval."""
    return set(_re.findall(r"^--- a/(.+)$", inst["patch"], _re.MULTILINE))

full_hit, partial, miss = [], [], []

for inst in pilot_instances:
    iid     = inst["instance_id"]
    gold    = gold_files(inst)
    found   = set(localized.get(iid, []))
    overlap = gold & found

    if overlap == gold:
        full_hit.append(iid)
    elif overlap:
        partial.append(iid)
    else:
        miss.append(iid)

n = len(pilot_instances)
print(f"Full    (all gold files found) : {len(full_hit)}/{n}  ({100*len(full_hit)/n:.0f}%)")
print(f"Partial (>=1 gold file found)  : {len(partial)}/{n}  ({100*len(partial)/n:.0f}%)")
print(f"Miss    (no gold files found)  : {len(miss)}/{n}  ({100*len(miss)/n:.0f}%)")

if partial:
    print("\nPartial:")
    for iid in partial:
        gold  = gold_files(inst_lookup[iid])
        found = set(localized[iid])
        print(f"  {iid}")
        print(f"    found : {sorted(found)}")
        print(f"    missed: {sorted(gold - found)}")

if miss:
    print("\nMiss:")
    for iid in miss:
        print(f"  {iid}: gold={sorted(gold_files(inst_lookup[iid]))}, got={localized.get(iid, [])}")


Full    (all gold files found) : 39/50  (78%)
Partial (>=1 gold file found)  : 9/50  (18%)
Miss    (no gold files found)  : 2/50  (4%)

Partial:
  django__django-12155
    found : ['django/contrib/admindocs/utils.py']
    missed: ['django/contrib/admindocs/views.py']
  pylint-dev__pylint-4604
    found : ['pylint/checkers/variables.py']
    missed: ['pylint/constants.py']
  django__django-11400
    found : ['django/contrib/admin/filters.py']
    missed: ['django/db/models/fields/__init__.py', 'django/db/models/fields/reverse_related.py']
  django__django-13195
    found : ['django/contrib/messages/storage/cookie.py', 'django/http/response.py']
    missed: ['django/contrib/sessions/middleware.py']
  matplotlib__matplotlib-24870
    found : ['lib/matplotlib/contour.py']
    missed: ['lib/matplotlib/tri/_tricontour.py']
  matplotlib__matplotlib-25479
    found : ['lib/matplotlib/cm.py', 'lib/matplotlib/pyplot.py']
    missed: ['lib/matplotlib/colors.py']
  django__django-13512
    found :

In [25]:
# Save localized files + hypotheses for reuse in other notebooks
import json
with open(f"localized_batch{BATCH}.json", "w") as f:
    json.dump(localized, f, indent=2)
with open(f"hypotheses_hctg_batch{BATCH}.json", "w") as f:
    json.dump(hypotheses, f, indent=2)
print(f"Saved → localized_batch{BATCH}.json")
print(f"Saved → hypotheses_hctg_batch{BATCH}.json")


Saved → localized_batch0.json
Saved → hypotheses_hctg_batch0.json


In [26]:
# ── Cell 7.5 — HCTG trace: selective trace + locus + fan-out ──────────────────
# HCTG Stages 3-5: walk the actual code from the issue entry point, find where
# expected (hypothesis) diverges from actual, then surface regression risks.
import json as _json


TRACE_SYSTEM = (
    "You have a hypothesis graph (expected flow) and the actual source code. "
    "Produce an ACTUAL-behavior trace graph, identify where expected and actual diverge, "
    "then identify regression risks if we change the divergence point.\n\n"
    "Output a single JSON object with this schema:\n\n"
    "{\n"
    "  \"actual_trace\": [\n"
    "    {\n"
    "      \"id\": \"n1\",\n"
    "      \"symbol\": \"path/to/file.py:ClassName.method\",\n"
    "      \"does\": \"what this symbol actually does, grounded in the real code\",\n"
    "      \"edges_out\": [\n"
    "        {\"to\": \"n2\", \"carries\": \"what is passed\", \"annotation\": \"[unexpected] | [black-box: reason] | null\"}\n"
    "      ]\n"
    "    }\n"
    "  ],\n"
    "  \"locus\": {\n"
    "    \"symbol\": \"path/to/file.py:ClassName.method\",\n"
    "    \"divergence_type\": \"missing_edge | wrong_value | wrong_order | early_return | absent_behavior\",\n"
    "    \"expected\": \"what the hypothesis predicted at this node\",\n"
    "    \"actual\": \"what the code actually does at this node\",\n"
    "    \"confidence\": \"high | medium | low\"\n"
    "  },\n"
    "  \"fan_out\": [\n"
    "    {\n"
    "      \"caller_symbol\": \"path:function\",\n"
    "      \"relies_on\": \"description of what current behavior it depends on\",\n"
    "      \"risk\": \"high | medium | low\"\n"
    "    }\n"
    "  ]\n"
    "}\n\n"
    "Rules:\n"
    "- `actual_trace` walks from the most plausible entry point for the bug described in the issue; "
    "only expand edges that could carry the failing condition. Edges that cannot affect the bug "
    "are `[black-box: reason]`.\n"
    "- `locus` is the FIRST node where expected ≠ actual — that is where the bug lives.\n"
    "- `fan_out` lists callers whose behavior depends on the locus's CURRENT (buggy) state — "
    "these are regression risks if we change the locus. Empty list if none detectable.\n"
    "- Use real symbol names `file.py:ClassName.method`, not role names.\n"
    "- Output ONLY the JSON object. No explanation before or after."
)


def generate_hctg_trace(inst):
    """HCTG Stages 3-5: produce actual trace, locus, and fan-out."""
    iid      = inst["instance_id"]
    repo_dir = get_repo_dir(iid)
    bc       = inst["base_commit"]
    files    = localized.get(iid, [])
    hyp      = hypotheses.get(iid) or ""

    code_blocks = []
    for fp in files:
        content = get_file_content(repo_dir, bc, fp)
        if content:
            code_blocks.append(f"### {fp}\n```python\n{content}\n```")

    if not code_blocks:
        return None, 0.0

    user_msg = (
        f"<issue>\n{inst['problem_statement']}\n</issue>\n\n"
        f"<hypothesis_graph>\n{hyp}\n</hypothesis_graph>\n\n"
        + "\n\n".join(code_blocks)
    )

    resp = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS, temperature=0.0,
        system=TRACE_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )

    raw = resp.content[0].text.strip()
    try:
        cleaned = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        graph = _json.loads(cleaned)
    except Exception:
        graph = None

    return {"raw": raw, "parsed": graph}, compute_cost(resp.usage)["total_cost_usd"]


# ── Run over all pilot instances ───────────────────────────────────────────────
# Cache-aware: if flowgraphs_hctg_batch{BATCH}.json exists and covers all pilot
# instances, load it instead of re-running. Delete the file to force re-run
# (e.g., after changing TRACE_SYSTEM).
from pathlib import Path

FLOWGRAPHS_CACHE = Path(f"flowgraphs_hctg_batch{BATCH}.json")

def _fg_cache_complete():
    if not FLOWGRAPHS_CACHE.exists():
        return False
    try:
        data = _json.load(open(FLOWGRAPHS_CACHE))
    except Exception:
        return False
    return all(i["instance_id"] in data for i in pilot_instances)

if _fg_cache_complete():
    flowgraphs = _json.load(open(FLOWGRAPHS_CACHE))
    total_graph_cost = 0.0
    parsed = sum(1 for v in flowgraphs.values() if v and v.get("parsed") is not None)
    print(f"Loaded cached flowgraphs for {len(pilot_instances)} instances (parsed: {parsed})")
    print(f"  (delete {FLOWGRAPHS_CACHE.name} to force re-run)")
else:
    flowgraphs       = {}
    total_graph_cost = 0.0
    parse_failures   = []

    for i, inst in enumerate(pilot_instances):
        iid = inst["instance_id"]
        print(f"[{i+1}/{len(pilot_instances)}] {iid} ... ", end="", flush=True)

        if not localized.get(iid):
            print("skipped (no localized files)")
            flowgraphs[iid] = None
            continue

        result, cost = generate_hctg_trace(inst)
        flowgraphs[iid]  = result
        total_graph_cost += cost

        if result is None:
            print("skipped (no files fetched)")
        elif result["parsed"] is None:
            parse_failures.append(iid)
            print(f"parse failed  ${cost:.4f}")
        else:
            locus = result["parsed"].get("locus", {}).get("symbol", "")[:60]
            print(f"ok  ${cost:.4f}  locus={locus}")

    print(f"\nHCTG trace done. Total cost: ${total_graph_cost:.4f}")
    print(f"Parsed: {sum(1 for v in flowgraphs.values() if v and v['parsed'] is not None)}/{len(pilot_instances)}")
    if parse_failures:
        print(f"Parse failures ({len(parse_failures)}): {parse_failures}")

    # Save flowgraphs to disk so the next iteration on prompt design can skip this stage
    import json
    with open(FLOWGRAPHS_CACHE, "w") as f:
        json.dump(
            {k: ({"raw": v["raw"], "parsed": v["parsed"]} if v else None)
             for k, v in flowgraphs.items()},
            f, indent=2, default=str
        )
    print(f"Saved → {FLOWGRAPHS_CACHE.name}")


Loaded cached flowgraphs for 50 instances (parsed: 50)
  (delete flowgraphs_hctg_batch0.json to force re-run)


In [27]:
# ── Cell 8 — Prompt builder ───────────────────────────────────────────────────

def _format_file(content, filepath):
    """Add line numbers so the model can reference exact lines in @@ headers."""
    numbered = "\n".join(f"{i+1} {line}" for i, line in enumerate(content.split("\n")))
    return f"[start of {filepath}]\n{numbered}\n[end of {filepath}]"

_PATCH_RULES = (
    "PATCH RULES:\n"
    "1. Context lines must be copied CHARACTER-FOR-CHARACTER from the snippet, "
    "including every leading space — do not paraphrase or change indentation.\n"
    "2. Use the line numbers shown at the start of each snippet line to set "
    "the correct @@ -N,M +N,M header.\n"
    "3. Never start or end a hunk with a blank context line — anchor on the "
    "nearest non-blank line above and below your change.\n"
    "4. Keep changes minimal. Do not add imports already present in the file.\n"
    "5. If <hctg> is present, it is a PRIOR hypothesis — not ground truth. Read the code first, "
    "form your own diagnosis, then cross-check against <hctg>. If the suggested locus is wrong, "
    "ignore it and fix the real bug. The listed callers are possibilities, not hard constraints — "
    "only preserve behavior that the actual code makes load-bearing.\n"
)

_PATCH_FORMAT = (
    "<patch>\n"
    "--- a/file.py\n"
    "+++ b/file.py\n"
    "@@ -1,27 +1,35 @@\n"
    " def euclidean(a, b):\n"
    "-    while b:\n"
    "-        a, b = b, a % b\n"
    "-    return a\n"
    "+    if b == 0:\n"
    "+        return a\n"
    "+    return euclidean(b, a % b)\n"
    "</patch>\n"
)


def _format_hctg(fg, hyp):
    """Build the <hctg> section from locus + fan_out only.

    Hypothesis and actual_trace are intentionally NOT injected at patch time:
    the model is reading the source code directly, so a narrative description
    of what the code does (trace) and a pre-code speculation about what it
    should do (hypothesis) are both noise that anchors the model to a
    potentially wrong diagnosis. Only the structured, actionable outputs
    survive: where to fix (locus) and what not to break (fan_out).
    """
    if not (fg and fg.get("parsed")):
        return ""
    parsed = fg["parsed"]
    locus  = parsed.get("locus") or {}
    fanout = parsed.get("fan_out") or []

    parts = []

    # Only surface the locus when we trust it. Low-confidence locus biases
    # without helping, so drop it entirely in that case.
    if locus.get("symbol") and locus.get("confidence") != "low":
        parts.append(
            f"SUGGESTED LOCUS (prior analysis — verify against code):\n"
            f"  symbol     : {locus.get('symbol','?')}\n"
            f"  divergence : {locus.get('divergence_type','?')}\n"
            f"  expected   : {locus.get('expected','?')}\n"
            f"  actual     : {locus.get('actual','?')}"
        )

    if fanout:
        fo_lines = [
            f"  - {r.get('caller_symbol','?')} [{r.get('risk','?')}]: {r.get('relies_on','?')}"
            for r in fanout
        ]
        parts.append("CALLERS THAT MAY BE AFFECTED:\n" + "\n".join(fo_lines))

    return "\n\n".join(parts)


def build_prompt(inst):
    iid      = inst["instance_id"]
    repo_dir = get_repo_dir(iid)
    bc       = inst["base_commit"]
    files    = localized.get(iid, [])

    code_blocks = []
    for fp in files:
        content = get_file_content(repo_dir, bc, fp)
        if content is not None:
            code_blocks.append(_format_file(content, fp))

    code_section = "\n\n".join(code_blocks) if code_blocks else "(no files retrieved)"

    hctg_body    = _format_hctg(flowgraphs.get(iid), hypotheses.get(iid))
    hctg_section = f"<hctg>\n{hctg_body}\n</hctg>\n\n" if hctg_body else ""

    return (
        "You will be provided with a partial code base and an issue statement "
        "explaining a problem to resolve.\n"
        f"<issue>\n{inst['problem_statement']}\n</issue>\n\n"
        f"{hctg_section}"
        f"<code>\n{code_section}\n</code>\n"
        f"{_PATCH_RULES}\n"
        "Output your patch in the following format:\n"
        f"{_PATCH_FORMAT}"
    )


In [28]:
# ── Cell 9 — Generate patches ─────────────────────────────────────────────────
import json
from pathlib import Path

results = []
out = Path(PREDICTIONS_FILE)
out.unlink(missing_ok=True)  # start fresh for this batch

for i, inst in enumerate(pilot_instances):
    iid = inst["instance_id"]
    print(f"[{i+1}/{len(pilot_instances)}] {iid} ... ", end="", flush=True)

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            temperature=0.0,
            messages=[{"role": "user", "content": build_prompt(inst)}],
        )
        cost   = compute_cost(response.usage)
        record = {
            "instance_id"        : iid,
            "model_name_or_path" : MODEL,
            "raw_response"       : response.content[0].text,
            "repo"               : inst["repo"],
            **cost,
        }
        print(f"${record['total_cost_usd']:.5f}  ({record['input_tokens']} in / {record['output_tokens']} out)")

    except Exception as e:
        print(f"ERROR: {e}")
        record = {
            "instance_id"        : iid,
            "model_name_or_path" : MODEL,
            "raw_response"       : "",
            "repo"               : inst["repo"],
            "error"              : str(e),
            "input_tokens": 0, "output_tokens": 0,
            "input_cost_usd": 0, "output_cost_usd": 0, "total_cost_usd": 0,
        }

    results.append(record)
    with open(out, "a") as f:
        f.write(json.dumps(record) + "\n")

total_cost = sum(r["total_cost_usd"] for r in results)
errors     = [r["instance_id"] for r in results if r.get("error")]
print(f"\nDone. Total cost: ${total_cost:.4f}")
if errors:
    print(f"Errors ({len(errors)}): {errors}")


[1/50] astropy__astropy-14995 ... $0.03725  (9990 in / 485 out)
[2/50] django__django-13810 ... $0.03167  (5241 in / 1063 out)
[3/50] sphinx-doc__sphinx-8595 ... $0.10785  (34359 in / 318 out)
[4/50] django__django-7530 ... $0.03062  (8378 in / 366 out)
[5/50] django__django-14631 ... $0.07270  (10372 in / 2772 out)
[6/50] django__django-14017 ... $0.08087  (21851 in / 1021 out)
[7/50] django__django-13837 ... $0.05570  (8402 in / 2033 out)
[8/50] django__django-12155 ... $0.01458  (3876 in / 197 out)
[9/50] pytest-dev__pytest-6202 ... $0.10959  (33296 in / 647 out)
[10/50] sphinx-doc__sphinx-10435 ... $0.13781  (39716 in / 1244 out)
[11/50] sphinx-doc__sphinx-8120 ... $0.06468  (19194 in / 473 out)
[12/50] scikit-learn__scikit-learn-13496 ... $0.13231  (41779 in / 465 out)
[13/50] django__django-13344 ... $0.06405  (10364 in / 2197 out)
[14/50] django__django-13121 ... $0.13399  (35043 in / 1924 out)
[15/50] pydata__xarray-3993 ... $0.42638  (138885 in / 648 out)
[16/50] pylint-dev__p

In [29]:
# ── Cell 10 — Extract patches ─────────────────────────────────────────────────
import json, re

def extract_best_patch(raw_response):
    """
    Extract the model's final patch from its raw response.
    Takes the last <patch> block per file — the model often outputs
    exploratory drafts before settling on its final answer.
    """
    patches = re.findall(r"<patch>(.*?)</patch>", raw_response, re.DOTALL)
    valid   = [p.strip() for p in patches if p.strip().startswith(("diff --git", "--- a/"))]
    if not valid:
        return ""

    file_to_idx = {}
    for idx, p in enumerate(valid):
        for f in re.findall(r"^--- a/(\S+)", p, re.MULTILINE):
            file_to_idx[f] = idx

    last_idx   = max(file_to_idx.values()) if file_to_idx else len(valid) - 1
    last_patch = valid[last_idx]
    last_files = set(re.findall(r"^--- a/(\S+)", last_patch, re.MULTILINE))

    if last_files == set(file_to_idx.keys()):
        return last_patch + "\n"

    seen, combined = set(), []
    for p in reversed(valid):
        for filepath, hunks in _parse_diff_blocks(p):
            if filepath not in seen:
                seen.add(filepath)
                combined.insert(0, (filepath, hunks))

    out = []
    for filepath, hunks in combined:
        out.append(f"--- a/{filepath}")
        out.append(f"+++ b/{filepath}")
        for hdr, body in hunks:
            out.append(hdr)
            out.extend(body)
    return "\n".join(out) + "\n"


def _parse_diff_blocks(patch_text):
    """Parse a unified diff into [(filepath, [(hunk_header, hunk_body)])]."""
    files = []
    cur_file = cur_hdr = None
    cur_hunks = []
    cur_body  = []
    for line in patch_text.split("\n"):
        if line.startswith("--- a/"):
            if cur_file:
                if cur_hdr is not None:
                    cur_hunks.append((cur_hdr, cur_body))
                files.append((cur_file, cur_hunks))
            cur_file = line[6:]; cur_hunks = []; cur_hdr = None; cur_body = []
        elif line.startswith("+++ b/"):
            pass
        elif line.startswith("@@ "):
            if cur_hdr is not None:
                cur_hunks.append((cur_hdr, cur_body))
            cur_hdr = line; cur_body = []
        elif cur_hdr is not None and line and line[0] in " -+":
            cur_body.append(line)
    if cur_file:
        if cur_hdr is not None:
            cur_hunks.append((cur_hdr, cur_body))
        files.append((cur_file, cur_hunks))
    return files


with open(PREDICTIONS_FILE) as fin, open(EXTRACTED_FILE, "w") as fout:
    for line in fin:
        pred  = json.loads(line.strip())
        iid   = pred["instance_id"]
        patch = extract_best_patch(pred.get("raw_response", ""))
        fout.write(json.dumps({
            "instance_id"        : iid,
            "model_patch"        : patch,
            "model_name_or_path" : MODEL,
        }) + "\n")

print(f"Written → {EXTRACTED_FILE}")


Written → predictions_hctg_batch0_extracted.jsonl


In [30]:
# ── Cell 11 — Fix hunk line numbers ───────────────────────────────────────────
import json, re

def fix_hunk(hdr, body, file_lines):
    """Re-anchor a hunk by searching for its context+removed lines in the actual file."""
    def parse_ops(lines):
        ops = []
        for line in lines:
            if line.startswith("-") and not line.startswith("---"):
                ops.append(("-", line[1:]))
            elif line.startswith("+") and not line.startswith("+++"):
                ops.append(("+", line[1:]))
            else:
                ops.append((" ", line[1:] if line.startswith(" ") else line))
        return ops

    def search_in_file(ops, file_lines):
        """Return 1-based start line if context+removed lines are found in file."""
        needle = [c for op, c in ops if op in (" ", "-")]
        if not needle or not file_lines:
            return None
        fs = [l.rstrip() for l in file_lines]
        ss = [l.rstrip() for l in needle]
        for i in range(len(fs) - len(ss) + 1):
            if fs[i : i + len(ss)] == ss:
                return i + 1
        return None

    ops   = parse_ops(body)
    found = search_in_file(ops, file_lines)

    if found is None:
        flipped = [("+", c) if op == "-" else ("-", c) if op == "+" else (" ", c) for op, c in ops]
        found   = search_in_file(flipped, file_lines)
        if found is not None:
            ops = flipped

    if found is None:
        m     = re.match(r"@@ -(\d+)", hdr)
        found = int(m.group(1)) if m else 1
    elif file_lines:
        offset, new_ops = 0, []
        for op, c in ops:
            if op in (" ", "-"):
                idx = found - 1 + offset
                new_ops.append((op, file_lines[idx] if idx < len(file_lines) else c))
                offset += 1
            else:
                new_ops.append((op, c))
        ops = new_ops

    orig_n  = sum(1 for op, _ in ops if op in (" ", "-"))
    new_n   = sum(1 for op, _ in ops if op in (" ", "+"))
    ctx     = re.search(r"@@[^@]*@@\s*(.*)", hdr)
    new_hdr = f"@@ -{found},{orig_n} +{found},{new_n} @@" + (f" {ctx.group(1)}" if ctx and ctx.group(1) else "")
    new_body = [f"{op}{c}" if op != " " else f" {c}" for op, c in ops]
    return new_hdr, new_body


def fix_patch_line_numbers(patch, iid, base_commit):
    """Fix @@ headers in every hunk using the real file content at base_commit."""
    repo_dir = get_repo_dir(iid)
    if not os.path.isdir(repo_dir):
        return patch
    blocks = _parse_diff_blocks(patch)
    if not blocks:
        return patch
    out = []
    for filepath, hunks in blocks:
        content    = get_file_content(repo_dir, base_commit, filepath)
        file_lines = content.splitlines() if content else None
        fixed_hunks = [fix_hunk(hdr, body, file_lines) if file_lines else (hdr, body) for hdr, body in hunks]
        fixed_hunks.sort(key=lambda hb: int(re.match(r"@@ -(\d+)", hb[0]).group(1)) if re.match(r"@@ -(\d+)", hb[0]) else 0)
        out += [f"--- a/{filepath}", f"+++ b/{filepath}"]
        for hdr, body in fixed_hunks:
            out.append(hdr)
            out.extend(body)
    return "\n".join(out) + "\n"


fixed_count = 0
with open(EXTRACTED_FILE) as fin, open(FIXED_FILE, "w") as fout:
    for line in fin:
        pred  = json.loads(line.strip())
        iid   = pred["instance_id"]
        patch = pred.get("model_patch", "")
        if patch and iid in inst_lookup:
            fixed = fix_patch_line_numbers(patch, iid, inst_lookup[iid]["base_commit"])
            if fixed != patch:
                fixed_count += 1
            patch = fixed
        fout.write(json.dumps({
            "instance_id"        : iid,
            "model_patch"        : patch,
            "model_name_or_path" : MODEL,
        }) + "\n")

print(f"Fixed {fixed_count} patches → {FIXED_FILE}")


Fixed 43 patches → predictions_hctg_batch0_fixed.jsonl


In [31]:
# ── Cell 12 — Evaluate ────────────────────────────────────────────────────────
import subprocess, json, os

VENV = "/tmp/swebench_venv"

def to_wsl_path(path):
    """Convert a Windows absolute path to a WSL /mnt/ path."""
    cwd   = os.getcwd()
    drive = cwd[0].lower()
    return "/mnt/" + drive + "/" + cwd[3:].replace(os.sep, "/") + "/" + path

def wsl(args, label=""):
    if label:
        print(f"  [{label}]")
    r = subprocess.run(
        ["wsl", "-d", "Ubuntu"] + args,
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    if r.stdout: print(r.stdout[-3000:])
    if r.stderr: print(r.stderr[-1000:])
    return r.returncode

def evaluate(predictions_file, run_id):
    """Run SWE-bench evaluation on a predictions file."""
    with open(predictions_file) as f:
        instance_ids = [json.loads(l)["instance_id"] for l in f]
    wsl([
        f"{VENV}/bin/python", "-m", "swebench.harness.run_evaluation",
        "--dataset_name", DATASET,
        "--predictions_path", to_wsl_path(predictions_file),
        "--instance_ids", *instance_ids,
        "--max_workers", "16",
        "--run_id", run_id,
    ], "evaluate")

print("Setting up venv...")
wsl(["python3", "-m", "venv", VENV], "venv")
wsl([f"{VENV}/bin/pip", "install", "swebench", "-q"], "pip install")

print(f"\n── Extracted predictions ──────────────────────────────────")
evaluate(EXTRACTED_FILE, f"{RUN_ID_BASE}_extracted")

print(f"\n── Fixed predictions ──────────────────────────────────────")
evaluate(FIXED_FILE, f"{RUN_ID_BASE}_fixed")

print("\nEvaluation complete.")


Setting up venv...
  [venv]
  [pip install]

── Extracted predictions ──────────────────────────────────
  [evaluate]
et-4-6/pylint-dev__pylint-4604/run_instance.log) for more information.
pylint-dev__pylint-4551: >>>>> Patch Apply Failed:
patching file pylint/pyreverse/diagrams.py
patch: **** malformed patch at line 17:                      names.append(node_name)


Check (logs/run_evaluation/hctg_batch0_extracted/claude-sonnet-4-6/pylint-dev__pylint-4551/run_instance.log) for more information.
pydata__xarray-3993: >>>>> Patch Apply Failed:
patching file xarray/core/dataarray.py
patch: **** malformed patch at line 11: @@ -3491,7 +3491,7 @@ class DataArray(AbstractArray, DataWithCoords):


Check (logs/run_evaluation/hctg_batch0_extracted/claude-sonnet-4-6/pydata__xarray-3993/run_instance.log) for more information.
django__django-14631: >>>>> Patch Apply Failed:
patching file django/forms/boundfield.py
patch: **** malformed patch at line 36:  @html_safe


Check (logs/run_evaluation/hctg

In [32]:
# ── Cell 13 — Parse results ───────────────────────────────────────────────────
import json, glob, os

def count_files(inst):
    return sum(1 for line in inst["patch"].splitlines() if line.startswith("--- a/"))

def load_report(run_id):
    path = f"{MODEL}.{RUN_ID_BASE}_{run_id}.json"
    if not os.path.exists(path):
        print(f"Not found: {path}")
        return None
    with open(path) as f:
        return json.load(f)

def print_report(report, label):
    if report is None:
        return
    total      = report["total_instances"]
    resolved   = report["resolved_instances"]
    errors     = report["error_instances"]
    resolved_ids    = set(report.get("resolved_ids", []))
    resolved_single = [iid for iid in resolved_ids if count_files(inst_lookup[iid]) == 1 if iid in inst_lookup]
    resolved_multi  = [iid for iid in resolved_ids if count_files(inst_lookup[iid]) >  1 if iid in inst_lookup]
    total_single    = [i for i in pilot_instances if count_files(i) == 1]
    total_multi     = [i for i in pilot_instances if count_files(i) >  1]

    print(f"{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Overall    : {resolved:>3} / {total}  ({100*resolved/total:.1f}%)")
    print(f"  Single file: {len(resolved_single):>3} / {len(total_single)}  ({100*len(resolved_single)/len(total_single):.1f}%)" if total_single else "  Single file: n/a")
    print(f"  Multi file : {len(resolved_multi):>3} / {len(total_multi)}  ({100*len(resolved_multi)/len(total_multi):.1f}%)" if total_multi else "  Multi file : n/a")
    print(f"  Errors     : {errors:>3} / {total}")
    print(f"{'='*50}")

    summary = {
        "notebook": "hctg", "model": MODEL, "batch": BATCH, "pipeline": label,
        "total": total, "resolved": resolved, "errors": errors,
        "resolve_rate": resolved / total if total else 0,
        "single_file": {"total": len(total_single), "resolved": len(resolved_single)},
        "multi_file":  {"total": len(total_multi),  "resolved": len(resolved_multi)},
    }
    out = RESULTS_FILE.replace(".json", f"_{label}.json")
    with open(out, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"  Saved → {out}")

print(f"Available reports: {glob.glob('*.json')}\n")
print_report(load_report("extracted"), "extracted")
print()
print_report(load_report("fixed"),     "fixed")


Available reports: ['claude-sonnet-4-6.agentless_batch0_clean.json', 'claude-sonnet-4-6.agentless_batch0_extracted.json', 'claude-sonnet-4-6.agentless_batch0_fixed.json', 'claude-sonnet-4-6.agentless_batch1_extracted.json', 'claude-sonnet-4-6.agentless_batch1_fixed.json', 'claude-sonnet-4-6.ast_batch1_extracted.json', 'claude-sonnet-4-6.ast_batch1_fixed.json', 'claude-sonnet-4-6.ast_flowgraph_batch0_extracted.json', 'claude-sonnet-4-6.ast_flowgraph_batch0_fixed.json', 'claude-sonnet-4-6.ast_flowgraph_batch1_extracted.json', 'claude-sonnet-4-6.ast_flowgraph_batch1_fixed.json', 'claude-sonnet-4-6.ast_graph_batch0_extracted.json', 'claude-sonnet-4-6.ast_graph_batch0_fixed.json', 'claude-sonnet-4-6.ast_graph_batch1_extracted.json', 'claude-sonnet-4-6.ast_graph_batch1_fixed.json', 'claude-sonnet-4-6.hctg_batch0_extracted.json', 'claude-sonnet-4-6.hctg_batch0_fixed.json', 'claude-sonnet-4-6.hctg_batch1_extracted.json', 'claude-sonnet-4-6.hctg_batch1_fixed.json', 'flowgraphs_hctg_batch0.json'

In [33]:
# ── Cell 14 — Localization coverage vs eval results ──────────────────────────
import re as _re

report = load_report("fixed")
if report is None:
    print("Run cell 13 first.")
else:
    resolved_ids = set(report.get("resolved_ids", []))

    rows = []
    for inst in pilot_instances:
        iid        = inst["instance_id"]
        gold       = set(_re.findall(r"^--- a/(.+)$", inst["patch"], _re.MULTILINE))
        found      = set(localized.get(iid, []))
        overlap    = gold & found
        coverage   = "full" if overlap == gold else "partial" if overlap else "miss"
        rows.append({"iid": iid, "passed": iid in resolved_ids, "coverage": coverage,
                     "gold": sorted(gold), "found": sorted(found), "missed": sorted(gold - found)})

    print(f"{'Coverage':<10}  {'Total':>6}  {'Passed':>8}  {'Failed':>8}  {'Pass rate':>10}")
    print("-" * 48)
    for cov in ("full", "partial", "miss"):
        subset = [r for r in rows if r["coverage"] == cov]
        passed = sum(r["passed"] for r in subset)
        rate   = f"{100*passed/len(subset):.0f}%" if subset else "—"
        print(f"{cov:<10}  {len(subset):>6}  {passed:>8}  {len(subset)-passed:>8}  {rate:>10}")
    print("-" * 48)
    total_pass = sum(r["passed"] for r in rows)
    print(f"{'total':<10}  {len(rows):>6}  {total_pass:>8}  {len(rows)-total_pass:>8}  {100*total_pass/len(rows):.0f}%")

    full_failed = [r for r in rows if r["coverage"] == "full" and not r["passed"]]
    if full_failed:
        print(f"\nFull coverage but FAILED ({len(full_failed)}) — patch generation issue:")
        for r in full_failed:
            print(f"  {r['iid']}")

    lucky = [r for r in rows if r["coverage"] != "full" and r["passed"]]
    if lucky:
        print(f"\nIncomplete coverage but PASSED ({len(lucky)}) — got lucky:")
        for r in lucky:
            print(f"  {r['iid']}  [{r['coverage']}]  missed: {r['missed']}")


Coverage     Total    Passed    Failed   Pass rate
------------------------------------------------
full            39        18        21         46%
partial          9         1         8         11%
miss             2         0         2          0%
------------------------------------------------
total           50        19        31  38%

Full coverage but FAILED (21) — patch generation issue:
  sphinx-doc__sphinx-8595
  django__django-14631
  django__django-14017
  sphinx-doc__sphinx-10435
  django__django-11490
  sphinx-doc__sphinx-7462
  django__django-14155
  django__django-11964
  astropy__astropy-14598
  pylint-dev__pylint-4551
  django__django-12663
  pytest-dev__pytest-10051
  pylint-dev__pylint-6528
  sympy__sympy-19783
  astropy__astropy-8707
  django__django-12406
  astropy__astropy-7606
  sympy__sympy-13974
  django__django-11333
  django__django-15103
  scikit-learn__scikit-learn-12973

Incomplete coverage but PASSED (1) — got lucky:
  django__django-12155  [partial]

In [34]:
# ── Regression analysis — HCTG vs AST flowgraph baseline ─────────────────────
import json, re as _re

def load_resolved_ids(run_id):
    """Load the set of resolved instance IDs from a SWE-bench report."""
    path = f"{MODEL}.{run_id}.json"
    if not os.path.exists(path):
        print(f"Not found: {path}")
        return set()
    with open(path) as f:
        report = json.load(f)
    return set(report.get("resolved_ids", []))

baseline_resolved = load_resolved_ids(f"ast_flowgraph_batch{BATCH}_fixed")
hctg_resolved     = load_resolved_ids(f"hctg_batch{BATCH}_fixed")

regressions  = baseline_resolved - hctg_resolved
improvements = hctg_resolved - baseline_resolved

n = len(pilot_instances)
print(f"AST flowgraph: {len(baseline_resolved)}/{n} resolved")
print(f"HCTG         : {len(hctg_resolved)}/{n} resolved")
print(f"Regressions  : {len(regressions)} (AST passed, HCTG failed)")
print(f"Improvements : {len(improvements)} (AST failed, HCTG passed)")

def locus_symbol(iid):
    fg = flowgraphs.get(iid)
    if fg and fg.get("parsed"):
        return fg["parsed"].get("locus", {}).get("symbol", "?")
    return "?"

print(f"\n── Regressions ──────────────────────────────────────────")
for iid in sorted(regressions):
    if iid not in inst_lookup:
        print(f"  {iid}  [not in inst_lookup — different batch]")
        continue
    multi = "multi" if count_files(inst_lookup[iid]) > 1 else "single"
    print(f"  {iid}  [{multi}]  locus={locus_symbol(iid)}")

print(f"\n── Improvements ─────────────────────────────────────────")
for iid in sorted(improvements):
    if iid not in inst_lookup:
        print(f"  {iid}  [not in inst_lookup]")
        continue
    multi = "multi" if count_files(inst_lookup[iid]) > 1 else "single"
    print(f"  {iid}  [{multi}]  locus={locus_symbol(iid)}")

# ── Deep dive: show the actual HCTG for regressions ─────────────────────────
print(f"\n── HCTG content for regressions ─────────────────────────")
for iid in sorted(regressions)[:3]:
    print(f"\n{'='*60}")
    print(f"  {iid}")
    print(f"{'='*60}")
    print(f"\n-- Hypothesis --")
    print((hypotheses.get(iid) or "[none]")[:800])
    fg = flowgraphs.get(iid)
    print(f"\n-- Trace/Locus/Fan-out --")
    if not fg:
        print("  [no trace generated]")
    elif not fg["parsed"]:
        print("  [parse failed — raw output:]")
        print(fg["raw"][:500])
    else:
        print(json.dumps(fg["parsed"], indent=2)[:2000])


AST flowgraph: 19/50 resolved
HCTG         : 19/50 resolved
Regressions  : 3 (AST passed, HCTG failed)
Improvements : 3 (AST failed, HCTG passed)

── Regressions ──────────────────────────────────────────
  django__django-14631  [multi]  locus=django/forms/forms.py:BaseForm._clean_fields
  django__django-15103  [multi]  locus=django/utils/html.py:json_script
  scikit-learn__scikit-learn-12973  [single]  locus=sklearn/linear_model/least_angle.py:LassoLarsIC.fit

── Improvements ─────────────────────────────────────────
  django__django-13121  [multi]  locus=django/db/models/expressions.py:DurationExpression.compile
  django__django-13810  [single]  locus=django/core/handlers/base.py:BaseHandler.load_middleware
  sympy__sympy-14711  [single]  locus=sympy/physics/vector/vector.py:Vector.__add__

── HCTG content for regressions ─────────────────────────

  django__django-14631

-- Hypothesis --
```
_clean_fields
  -> iterate_bound_items [passes: name, BoundField]
  -> bf_get_value [passes: